# Day 34 — Advanced SQL: Recursive CTEs + Hierarchies
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Write recursive CTEs in SQLite to traverse org hierarchies
2. Calculate hierarchy depth, path enumeration, and all-descendant queries
3. Connect the pattern to SAP HR org structure (OTYPE, SOBID in HRP1001)

---
> **SAP Context:** SAP HCM stores org structure in table HRP1001 (relationship table). Each object (org unit, position, job) has an OTYPE (e.g., 'O' = org unit, 'S' = position, 'P' = person). Relationships like 'belongs to' (A002) and 'is line supervisor of' (A002) form the hierarchy. Recursive CTEs replicate the hierarchy traversal that SAP's P_ORGXX functions perform in ABAP.


## 0. Setup — Build a Hierarchy Table

In [ ]:
import sqlite3, pandas as pd
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
con = sqlite3.connect(':memory:')

# Load HR data
hr = pd.read_csv(DATA / 'hr_headcount.csv')
hr.to_sql('hr', con, if_exists='replace', index=False)

# Build a simulated org hierarchy
# In a real SAP extract, this would come from HRP1001
# Here: create ORG_HIERARCHY with parent-child relationships between ORGEH codes

import pandas as pd
import numpy as np

# Create a simple hierarchy from the org units present in HR
orgs = hr[['ORGEH','ORGTX']].drop_duplicates().sort_values('ORGEH').reset_index(drop=True)

# Simulate parent relationships: assign parents deterministically
# (In real SAP this would come from HRP1001)
np.random.seed(42)
n = len(orgs)

def assign_parent(i, n):
    if i == 0: return None  # root
    # Parent is ~30% chance of being 2 levels up, otherwise 1 level up
    parent_idx = max(0, i - np.random.choice([1,1,2]))
    return orgs.iloc[parent_idx]['ORGEH']

orgs['PARENT_ORGEH'] = [assign_parent(i, n) for i in range(n)]
orgs.to_sql('org_hierarchy', con, if_exists='replace', index=False)

def q(sql): return pd.read_sql_query(sql, con)

print(f'Org hierarchy: {len(orgs)} nodes')
print(orgs.head(8).to_string(index=False))

## 1. Basic Recursive CTE — Full Hierarchy Traversal

In [ ]:
# YOUR SOLUTION
# Traverse from root to all descendants, calculating depth

recursive_hierarchy = '''
WITH RECURSIVE org_tree AS (
    -- Anchor: root nodes (no parent)
    SELECT
        ORGEH,
        ORGTX,
        PARENT_ORGEH,
        0 AS depth,
        CAST(ORGEH AS TEXT) AS path
    FROM org_hierarchy
    WHERE PARENT_ORGEH IS NULL

    UNION ALL

    -- Recursive: join children to current level
    SELECT
        child.ORGEH,
        child.ORGTX,
        child.PARENT_ORGEH,
        parent.depth + 1,
        parent.path || ' > ' || CAST(child.ORGEH AS TEXT)
    FROM org_hierarchy child
    JOIN org_tree parent ON child.PARENT_ORGEH = parent.ORGEH
)
SELECT
    ORGEH,
    ORGTX,
    PARENT_ORGEH,
    depth,
    path
FROM org_tree
ORDER BY path
'''

tree = q(recursive_hierarchy)
print('=== Full Org Hierarchy ===')
for _, row in tree.iterrows():
    indent = '  ' * row['depth']
    print(f"{indent}[{row['ORGEH']}] {row['ORGTX']}  (depth={row['depth']})")

## 2. Find All Direct and Indirect Reports for a Manager

In [ ]:
# YOUR SOLUTION
# Given a root org unit, find all org units underneath it (the subtree)
# This is the SAP P_ORGXX equivalent: "give me everyone under this org"

ROOT_ORG = orgs[orgs['PARENT_ORGEH'].isna()]['ORGEH'].iloc[0]

subtree_sql = f'''
WITH RECURSIVE subtree AS (
    -- Start at the target org unit
    SELECT ORGEH, ORGTX, PARENT_ORGEH, 0 AS depth
    FROM org_hierarchy
    WHERE ORGEH = '{ROOT_ORG}'

    UNION ALL

    -- Recursively add children
    SELECT child.ORGEH, child.ORGTX, child.PARENT_ORGEH, parent.depth + 1
    FROM org_hierarchy child
    JOIN subtree parent ON child.PARENT_ORGEH = parent.ORGEH
)
SELECT s.ORGEH, s.ORGTX, s.depth,
       COUNT(h.PERNR) AS headcount,
       ROUND(AVG(h.SALARY), 0) AS avg_salary
FROM subtree s
LEFT JOIN hr h ON s.ORGEH = h.ORGEH AND h.TERM_DATE IS NULL
GROUP BY s.ORGEH, s.ORGTX, s.depth
ORDER BY s.depth, s.ORGEH
'''

subtree_df = q(subtree_sql)
print(f'=== Subtree under root org {ROOT_ORG} ===')
print(subtree_df.to_string(index=False))

## 3. Path Enumeration — Full Ancestry of Each Node

In [ ]:
# YOUR SOLUTION
# For each org unit, show the complete path from root: A > B > C > D

path_sql = '''
WITH RECURSIVE ancestry AS (
    SELECT
        ORGEH AS target,
        ORGEH AS current_node,
        ORGTX AS current_name,
        PARENT_ORGEH,
        1 AS level,
        CAST(ORGEH AS TEXT) AS path_nodes
    FROM org_hierarchy

    UNION ALL

    SELECT
        a.target,
        p.ORGEH,
        p.ORGTX,
        p.PARENT_ORGEH,
        a.level + 1,
        CAST(p.ORGEH AS TEXT) || ' > ' || a.path_nodes
    FROM ancestry a
    JOIN org_hierarchy p ON p.ORGEH = a.PARENT_ORGEH
)
-- Only keep the root-to-node full path (where we've reached the root)
SELECT target AS ORGEH, path_nodes AS full_path, level AS depth
FROM ancestry
WHERE PARENT_ORGEH IS NULL
ORDER BY target
LIMIT 15
'''

paths = q(path_sql)
print('=== Full Ancestry Paths ===')
print(paths.to_string(index=False))

## 4. Level Calculation + SAP Context

In [ ]:
# YOUR SOLUTION
# Count nodes at each depth level and show headcount by level

level_sql = '''
WITH RECURSIVE tree AS (
    SELECT ORGEH, ORGTX, PARENT_ORGEH, 0 AS depth
    FROM org_hierarchy WHERE PARENT_ORGEH IS NULL
    UNION ALL
    SELECT c.ORGEH, c.ORGTX, c.PARENT_ORGEH, t.depth + 1
    FROM org_hierarchy c JOIN tree t ON c.PARENT_ORGEH = t.ORGEH
)
SELECT depth AS hierarchy_level,
       COUNT(DISTINCT t.ORGEH) AS org_unit_count,
       COUNT(h.PERNR) AS total_headcount
FROM tree t
LEFT JOIN hr h ON t.ORGEH = h.ORGEH AND h.TERM_DATE IS NULL
GROUP BY depth
ORDER BY depth
'''

levels = q(level_sql)
print('=== Headcount by Hierarchy Level ===')
print(levels.to_string(index=False))
print()
print('''SAP HRP1001 mapping:
  OTYPE = 'O'  → Org Unit  (this table)
  OTYPE = 'S'  → Position
  OTYPE = 'P'  → Person
  RELAT = 'A002' → "belongs to" (child → parent org unit)
  RELAT = 'B002' → "incorporates" (parent → child org unit)
  
  The recursive CTE above replicates what SAP's RHFILL00 report
  and the P_ORGXX authorization objects do during authority checks.
''')

## Daily Prompt
**Answer independently:**

1. Extend the subtree query to show, for each org unit in the subtree, the number of levels to the nearest leaf node (lowest level with no children). This is the "depth remaining" calculation.
2. In SAP HCM, what is the difference between ORGEH (org unit key in the HR module) and KOSTL (cost center in CO)? Are they always 1-to-1? What happens at the boundary between HR and CO in SAP?
3. Write a recursive CTE that detects circular references in the org hierarchy (a situation where org A is parent of org B, but org B is also in the ancestry of org A). How would you prevent this from causing an infinite loop?
